# Eskom 2023 Parser Report

This notebook reads the canonical module 02 CSV outputs and summarizes the parser repair, timestamp filter, target anchors, and accounting checks used by the South Africa fixed-validation workflow.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/pypsa-earth-matplotlib")

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = Path.cwd().parents[2]

HOURLY = ROOT / "data/za_validation/eskom_2023_hourly_clean.csv"
TARGETS = ROOT / "data/za_validation/eskom_2023_targets_by_carrier.csv"
REPORT = ROOT / "data/za_audit/eskom_2023_parser_report.csv"

hourly = pd.read_csv(HOURLY, parse_dates=["Date Time Hour Beginning"])
targets = pd.read_csv(TARGETS)
report = pd.read_csv(REPORT)

{
    "hourly_rows": len(hourly),
    "targets": len(targets),
    "report_checks": len(report),
    "start": hourly["Date Time Hour Beginning"].min(),
    "end": hourly["Date Time Hour Beginning"].max(),
}


## Column Inspection

The parser stores the raw header list in `data/za_audit/eskom_2023_parser_report.csv`. These columns are the basis for the resolved accounting identity.

In [ ]:
header_value = report.loc[report["check_id"].eq("raw-column-headers"), "value"].iloc[0]
headers = header_value.split(" | ")
pd.DataFrame({"position": range(len(headers)), "column": headers})


## Parser Accounting

Rows with a split `Total UCLF+OCLF` comma decimal are repaired before numeric parsing. The 2023 slice must contain exactly 8,760 hourly observations.

In [ ]:
parser_checks = report[report["category"].isin(["parser", "filter", "schema", "provenance"])]
display(parser_checks[["check_id", "status", "value", "unit", "notes"]])

assert len(hourly) == 8760
assert hourly["Date Time Hour Beginning"].is_unique
assert hourly["Date Time Hour Beginning"].min() == pd.Timestamp("2023-01-01 00:00")
assert hourly["Date Time Hour Beginning"].max() == pd.Timestamp("2023-12-31 23:00")


## Accounting Checks

The resolved identity is `Residual Demand = Dispatchable Generation + Manual Load_Reduction(MLR) + ILS Usage + IOS Excl ILS and MLR`. `RSA Contracted Demand - Residual Demand - Total RE` is retained as a source discrepancy warning.

In [ ]:
accounting = report[report["category"].eq("accounting")]
display(accounting[["check_id", "status", "value", "unit", "tolerance", "notes"]])

blocking_failures = accounting[accounting["status"].eq("fail")]
assert blocking_failures.empty


## Annual Targets

Energy targets are annual TWh totals computed from hourly MW averages. Capacity targets use the end-of-year 2023 value where later validation needs one reference capacity.

In [ ]:
display(targets[["target", "value", "unit", "status", "source", "notes"]])

required_targets = {
    "RSA Contracted Demand",
    "Residual Demand",
    "Dispatchable Generation",
    "Manual Load Reduction",
    "ILS Usage",
    "IOS Excl ILS and MLR",
    "MLR + ILS + IOS",
    "International Imports",
    "International Exports",
    "Eskom Gas Generation",
}
missing = required_targets.difference(set(targets["target"]))
assert not missing, missing


## Time-Series View

The plots below provide a fast visual check that demand, renewable generation, and reduced/unserved-demand components are in the expected order of magnitude.

In [ ]:
plot_df = hourly.set_index("Date Time Hour Beginning")
daily = plot_df.resample("D").mean()
daily_reduced = plot_df[["Manual Load_Reduction(MLR)", "ILS Usage", "IOS Excl ILS and MLR"]].resample("D").sum() / 1000

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
daily[["RSA Contracted Demand", "Residual Demand", "Dispatchable Generation"]].plot(ax=axes[0], linewidth=1.3)
axes[0].set_ylabel("MW")
axes[0].set_title("Daily average demand and dispatchable generation")

daily[["Wind", "PV", "CSP", "Other RE"]].plot(ax=axes[1], linewidth=1.2)
axes[1].set_ylabel("MW")
axes[1].set_title("Daily average renewable generation")

daily_reduced.plot(ax=axes[2], linewidth=1.2)
axes[2].set_ylabel("GWh/day")
axes[2].set_title("Daily reduced/unserved demand components")

for ax in axes:
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper left", fontsize=8)

fig.tight_layout()
plt.show()


## Status

Module 02 parser outputs are valid when the row-count assertions pass, accounting checks have no `fail` rows, and the only expected accounting warning is the retained `RSA Contracted Demand - Residual Demand - Total RE` discrepancy.